# Redução de Dimensionalidade das Métricas Locais via PCA

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, ttest_ind, ttest_rel, wilcoxon

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

### Carregando o dataset

In [2]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape
df_filtered = df_dataset.columns[df_dataset.columns.str.contains("age", case=False)]
df_filtered


Index(['age', 'age_25-30', 'age_30-35', 'age_55-60', 'age_60-65', 'age_65-70',
       'age_70-75', 'age_75-80'],
      dtype='object')

### Seleção Métricas Locais

In [3]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

NODE_GROUPS = {
    "degree": [c for c in df_dataset.columns if c.startswith("degree_")],
    "clustering": [c for c in df_dataset.columns if c.startswith("clustering_")],
    "betweenness": [c for c in df_dataset.columns if c.startswith("betweenness_")],
}


len(NODE_FEATURES)

212

In [4]:
# descarta colunas sem informação (constantes ou duplicadas), se houver
DROP_REDUNDANT = True

_raw = df_dataset[NODE_FEATURES]
constant_cols = [c for c in NODE_FEATURES if _raw[c].nunique(dropna=True) <= 1]
_dup = _raw.T.duplicated()
duplicate_cols = [c for c, d in zip(NODE_FEATURES, _dup) if d and c not in constant_cols]

if DROP_REDUNDANT:
    NODE_FEATURES = [c for c in NODE_FEATURES if c not in constant_cols and c not in duplicate_cols]

print(f"Constantes: {len(constant_cols)} | duplicadas: {len(duplicate_cols)} | usadas no PCA: {len(NODE_FEATURES)}")
print("Tipos constantes:", sorted({c.rsplit("_", 1)[0] for c in constant_cols}))
print("Tipos duplicados:", sorted({c.rsplit("_", 1)[0] for c in duplicate_cols}))

Constantes: 0 | duplicadas: 0 | usadas no PCA: 212
Tipos constantes: []
Tipos duplicados: []


In [5]:
X = df_dataset[NODE_FEATURES]
y = df_dataset["condition"].values
groups = df_dataset["subject_id"].values

print(f"{len(X)} registros de {df_dataset['subject_id'].nunique()} participantes (EO e EC de cada um)")

108 registros de 54 participantes (EO e EC de cada um)


### Padronização das Feaures

Imputação pela mediana: nem todo participante tem todos os eletrodos. Valor ausente significa eletrodo ausente, não conectividade zero, por isso não se usa 0.

In [6]:
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [8]:
X_scaled.mean(axis=0)[:5], X_scaled.std(axis=0)[:5]

(array([ 3.60822483e-16,  6.26042428e-16, -4.31753398e-17, -2.52884133e-16,
         4.52313084e-17]),
 array([1., 1., 1., 1., 1.]))

### PCA exploratório

In [9]:
print("NaN antes da imputação:", int(df_dataset[NODE_FEATURES].isna().sum().sum()))
print("NaN depois da imputação:", int(np.isnan(X).sum()))

NaN antes da imputação: 224
NaN depois da imputação: 0


In [10]:
nan_per_feature = df_dataset[NODE_FEATURES].isna().sum()
nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)

hub_frequency_CP6    8
betweenness_CP6      8
clustering_CP6       8
degree_CP6           8
degree_CP1           6
                    ..
hub_frequency_C4     1
degree_F6            1
clustering_F6        1
betweenness_F6       1
hub_frequency_F6     1
Length: 80, dtype: int64

In [11]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

In [12]:
df_var = pd.DataFrame({
    "component": np.arange(1, len(explained_var) + 1),
    "explained_variance": explained_var,
    "cumulative_variance": cum_var
})

In [13]:
df_var

,component,explained_variance,cumulative_variance
0,1,5.284992e-01,0.528499
1,2,6.562737e-02,0.594127
2,3,3.508946e-02,0.629216
3,4,2.692351e-02,0.656139
4,5,2.268602e-02,0.678826
...,...,...,...
103,104,3.219916e-05,0.999936
104,105,2.571870e-05,0.999962
105,106,2.301460e-05,0.999985
106,107,1.478067e-05,1.000000


In [14]:
n80 = int(np.argmax(cum_var >= 0.8) + 1)
n90 = int(np.argmax(cum_var >= 0.9) + 1)
print(f"PC1: {explained_var[0]:.1%} | PC2: {explained_var[1]:.1%}")
print(f"Componentes para 80%: {n80} | para 90%: {n90}")

PC1: 52.8% | PC2: 6.6%
Componentes para 80%: 14 | para 90%: 29


In [15]:
fig = px.line(
    df_var,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="Variância Acumulada Explicada pelo PCA"
)
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%")
fig.add_hline(y=0.9, line_dash="dash", annotation_text="90%")
fig.show()

Observa-se que 14 componentes principais são suficientes para explicar 80% da variância total dos dados, enquanto 29 componentes explicam 90%, indicando elevada redundância entre as métricas de rede (212 variáveis: grau ponderado normalizado, clustering ponderado, intermediação com distância 1/peso e frequência de hub, em 53 eletrodos).

### PCA com 2 componentes

In [16]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURES_BY_TYPE = {
    prefix.rstrip("_"): [c for c in NODE_FEATURES if c.startswith(prefix)]
    for prefix in NODE_FEATURE_PREFIXES
}


In [17]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

df_pca = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)
df_pca["condition"] = y
df_pca["subject_id"] = groups

In [18]:
fig = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="condition",
    title="Projeção PCA das Métricas Locais (2 Componentes)",
    opacity=0.8
)
fig.show()

Leitura visual do gráfico:

existe separação EO × EC? Não há separação clara entre EO e EC

existe sobreposição? Sim

clusters? Existem regiões de maior densidade, mas não clusters bem definidos.

A projeção das métricas locais no espaço definido pelos dois primeiros
componentes principais não evidenciou uma separação clara entre as
condições EO e EC. Observa-se uma ampla sobreposição entre os pontos
correspondentes às duas condições experimentais, indicando que os
padrões dominantes de variabilidade capturados pelos componentes
principais são compartilhados entre os grupos.

Embora seja possível identificar regiões de maior densidade de pontos,
não foram observados clusters bem definidos ou fronteiras visuais que
permitam distinguir claramente as condições experimentais no espaço
PC1–PC2.


### Contribuição das features

In [19]:
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=NODE_FEATURES,
    columns=["PC1", "PC2"]
)

In [20]:
loadings

,PC1,PC2
degree_AF3,0.083467,0.032919
clustering_AF3,0.088496,0.027587
betweenness_AF3,-0.039281,0.161480
hub_frequency_AF3,-0.031707,-0.037032
degree_AF4,0.082684,0.025171
...,...,...
hub_frequency_Pz,-0.053358,0.016691
degree_TP8,0.086487,0.053347
clustering_TP8,0.090367,0.040574
betweenness_TP8,-0.044162,0.092859


In [21]:
loadings.abs().sort_values("PC1", ascending=False).head(10)

,PC1,PC2
clustering_P3,0.093416,0.020873
clustering_P4,0.093356,0.025530
clustering_P1,0.093296,0.019265
clustering_P2,0.093226,0.019090
clustering_PO3,0.093222,0.025655
clustering_P6,0.093166,0.028201
clustering_PO4,0.093163,0.026179
clustering_POz,0.093154,0.025204
clustering_P5,0.093127,0.024533
clustering_Pz,0.093067,0.020263


In [22]:
loadings.abs().sort_values("PC2", ascending=False).head(10)

,PC1,PC2
betweenness_P7,0.038258,0.229271
betweenness_CP1,0.040447,0.227233
betweenness_C1,0.041321,0.226154
betweenness_Cz,0.035218,0.224353
betweenness_FC2,0.038629,0.219913
betweenness_P1,0.046229,0.212012
betweenness_CP4,0.044049,0.207303
betweenness_Oz,0.049364,0.205104
betweenness_P8,0.042625,0.198918
betweenness_C2,0.047648,0.195749


#### Cargas por tipo de métrica

In [23]:
_lt = loadings[["PC1", "PC2"]].assign(tipo=[c.rsplit("_", 1)[0] for c in loadings.index])

# cada componente tem norma 1: a soma das cargas ao quadrado de um tipo é a fração do componente formada por ele
loadings_by_type = _lt.groupby("tipo").agg(
    n_variaveis=("PC1", "size"),
    PC1_contribuicao=("PC1", lambda s: (s ** 2).sum()),
    PC2_contribuicao=("PC2", lambda s: (s ** 2).sum()),
    PC1_carga_media=("PC1", "mean"),
    PC1_frac_positivas=("PC1", lambda s: (s > 0).mean()),
).sort_values("PC1_contribuicao", ascending=False)

fig = px.bar(
    loadings_by_type.reset_index().melt(
        id_vars="tipo", value_vars=["PC1_contribuicao", "PC2_contribuicao"],
        var_name="componente", value_name="contribuicao",
    ),
    x="tipo", y="contribuicao", color="componente", barmode="group",
    title="Contribuição de cada tipo de métrica para PC1 e PC2",
    labels={"tipo": "Métrica", "contribuicao": "Soma das cargas²", "componente": "Componente"},
)
fig.show()

loadings_by_type.round(3)

,n_variaveis,PC1_contribuicao,PC2_contribuicao,PC1_carga_media,PC1_frac_positivas
tipo,,,,,
clustering,53,0.435,0.032,0.091,1.000
degree,53,0.403,0.040,0.087,1.000
betweenness,53,0.092,0.825,-0.040,0.000
hub_frequency,53,0.071,0.103,-0.034,0.019


Quais métricas e nós mais contribuem para cada componente?

PC1 explica 52,8% da variância e PC2, 6,6%. PC1 é formado principalmente por clustering (≈44% da soma das cargas²) e grau (≈40%), com cargas positivas em todos os eletrodos; intermediação (≈9%) e frequência de hub (≈7%) entram com cargas negativas em quase todos. É um eixo de nível geral de sincronização: participantes cujos eletrodos passam mais tempo sincronizados têm também triângulos mais intensos, caminhos mais distribuídos (menos intermediação concentrada) e menos eletrodos se destacando como hubs. PC2 é dominado pela intermediação (≈83%; frequência de hub ≈10%) e capta o quanto os caminhos da rede se concentram em alguns eletrodos.

### Número de componentes para 90% da variância

In [24]:
pca_n = PCA(n_components=0.9)  # mantém 90% da variância
X_pca_n = pca_n.fit_transform(X_scaled)

X_pca_n.shape

(108, 29)

Com todos os 108 registros, 34 componentes retêm 90% da variância. Essa projeção é só descritiva (mede a redundância entre as métricas) e não é usada nos modelos: nas classificações, imputação, padronização e PCA são ajustados dentro de cada dobra da validação cruzada, só com os dados de treino.

A aplicação do PCA sobre as métricas locais permitiu reduzir significativamente
a dimensionalidade do conjunto de dados, preservando a maior parte da variância
original.

A projeção bidimensional não separa EO e EC: os pontos das duas condições se sobrepõem quase totalmente, porque a variação entre participantes é maior que a diferença entre condições.

As cargas mostram que PC1 é um eixo global, e não regional: grau e clustering entram com cargas positivas em todos os eletrodos. PC2 capta como a frequência de hub e a intermediação se distribuem entre regiões.


-----

### Teste Estatístico

EO e EC são do mesmo participante, então a comparação de PC1 é **pareada** por `subject_id` (teste t pareado e Wilcoxon), e não entre grupos independentes.

In [25]:
pc1_wide = df_pca.pivot(index="subject_id", columns="condition", values="PC1").dropna()
pc1_eo, pc1_ec = pc1_wide["EO"], pc1_wide["EC"]

print(f"Pares completos: {len(pc1_wide)}")
print(f"Mediana da diferença EO - EC: {(pc1_eo - pc1_ec).median():.3f}")
ttest_rel(pc1_eo, pc1_ec)

Pares completos: 54
Mediana da diferença EO - EC: -2.428


TtestResult(statistic=np.float64(-3.572743129696683), pvalue=np.float64(0.0007621025507883991), df=np.int64(53))

In [26]:
fig = px.line(
    df_pca.sort_values("condition", ascending=False),
    x="condition", y="PC1", line_group="subject_id", markers=True,
    title="PC1 por participante: EO vs EC",
    category_orders={"condition": ["EO", "EC"]},
)
fig.update_traces(opacity=0.4)
fig.show()

In [27]:
wilcoxon(pc1_eo, pc1_ec)

WilcoxonResult(statistic=np.float64(375.0), pvalue=np.float64(0.0015548044386322517))

In [28]:
px.box(df_pca, x="condition", y="PC1")

In [29]:
px.box(df_pca, x="condition", y="PC2")

In [30]:
df_pca = df_pca[df_pca["condition"].isin(["EO", "EC"])]

In [31]:
cv_groups = GroupKFold(n_splits=5)

# PCA ajustado dentro de cada dobra; EO e EC do mesmo participante ficam sempre na mesma dobra
model_pca2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("clf", LogisticRegression()),
])

scores = cross_val_score(
    model_pca2, df_dataset[NODE_FEATURES], df_dataset["condition"],
    cv=cv_groups, groups=groups,
)
print(scores.round(3))
scores.mean()

[0.545 0.545 0.727 0.591 0.6  ]


np.float64(0.6018181818181818)

Com validação cruzada agrupada por participante (GroupKFold, 5 dobras) e o PCA ajustado dentro de cada dobra, a regressão logística com PC1 e PC2 obteve acurácia média de 60,2% (dobras entre 55% e 73%), acima do acaso (50%), mas modesta. As duas primeiras componentes carregam alguma informação sobre a condição, com capacidade discriminativa limitada.

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

X = df_dataset[NODE_FEATURES]
y = df_dataset["condition"]

model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,  # equivalente a L1 (penalty='l1' deprecated)
                max_iter=5000,
            ),
        ),
    ]
)

In [33]:
scores = cross_val_score(model, X, y, cv=cv_groups, groups=groups)
print(scores.round(3))
scores.mean()

[0.591 0.682 0.636 0.682 0.6  ]


np.float64(0.6381818181818182)

Os dois primeiros componentes principais resumem boa parte da variabilidade das métricas de rede, mas não separam visualmente EO e EC no plano PC1–PC2, porque a variação entre participantes é muito maior que a diferença entre condições. Como EO e EC vêm dos mesmos 54 participantes, a comparação de PC1 foi feita de forma pareada: PC1 foi menor em EO do que em EC em 37 dos 54 participantes (mediana da diferença EO − EC = −2,43; teste t pareado: p < 0,001; Wilcoxon: p = 0,002). Ou seja, dentro de cada pessoa, o nível geral de sincronização tende a ser menor com olhos abertos. Entre pessoas, essa diferença permite apenas uma classificação modesta: com validação agrupada por participante, a regressão logística com PC1 e PC2 teve acurácia de 60,2%, e o modelo L1 com todas as métricas, 63,8%. A diferença EO − EC também depende da idade (ver `04_pca_metadata`).